# Analyse des concessions

Objectif : comparer le loyer contractuel (`sRent`) au loyer effectif (`sRentEffective`), puis expliquer leur divergence de croissance à partir de 2024. Les renouvellements et les relocations sont inclus, puis examinés séparément.

Le loyer effectif fourni tient compte des concessions, y compris certains avantages comme le stationnement gratuit. Il représente un montant ajusté des avantages et non une vérification des encaissements. Nous utilisons cette colonne sans additionner à nouveau les concessions, ce qui compterait les rabais deux fois.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 20)
plt.rcParams.update({"figure.figsize": (10, 5), "axes.grid": True})
leases = pd.read_csv(Path("equinoxe_lease_history.csv"), dtype={"sPropCode": str, "sUnitCode": str})
for col in ["sLeaseFrom", "sLeaseTo"]:
    leases[col] = pd.to_datetime(leases[col], errors="coerce")
for col in ["sRent", "sRentEffective", "sConcession", "sRenewal"]:
    leases[col] = pd.to_numeric(leases[col], errors="coerce")
leases["annee"] = leases["sLeaseFrom"].dt.year
leases["type_bail"] = leases["sRenewal"].map({0: "Relocation", 1: "Renouvellement"}).fillna("Non renseigné")
display(leases[["hUnit", "sBuilding", "sLeaseFrom", "sRent", "sRentEffective", "sConcession", "type_bail"]].head())

## 1. Vérifier les données

Nous conservons uniquement les baux datés dont les deux loyers sont strictement positifs. Les comparaisons utilisent `hUnit`, l'identifiant de l'appartement. Une date de début doit désigner un seul bail par appartement : des doublons ambigus doivent être examinés avant de poursuivre.

In [ ]:
controle = pd.Series({
    "Nombre de lignes": len(leases),
    "Dates de début manquantes": leases.sLeaseFrom.isna().sum(),
    "Identifiants d'appartement manquants": leases.hUnit.isna().sum(),
    "Loyers contractuels manquants ou non positifs": (~leases.sRent.gt(0)).sum(),
    "Loyers effectifs manquants ou non positifs": (~leases.sRentEffective.gt(0)).sum(),
    "Loyers effectifs supérieurs aux contractuels": leases.sRentEffective.gt(leases.sRent + 0.01).sum(),
    "Doublons exacts": leases.duplicated().sum(),
}, name="Nombre")
display(controle.to_frame())
base = leases.drop_duplicates().copy()
ambigus = base.duplicated(["hUnit", "sLeaseFrom"], keep=False)
if ambigus.any():
    display(base.loc[ambigus, ["hUnit", "sLeaseFrom", "sRent", "sRentEffective"]])
    raise ValueError("Plusieurs baux pour le même appartement à la même date : examiner ces lignes.")
base["valide"] = (base.hUnit.notna() & base.sLeaseFrom.notna()
                   & base.sRent.gt(0) & base.sRentEffective.gt(0))
valides = base.loc[base.valide].copy()
valides["rabais_mensuel"] = valides.sRent - valides.sRentEffective
valides["rabais_pct"] = 100 * valides.rabais_mensuel / valides.sRent
valides["avec_rabais"] = valides.rabais_mensuel > 0.01
periode = valides.loc[valides.annee.between(2021, 2025)].copy()
print(f"Baux valides : {len(valides):,} / {len(base):,}")
print("Écarts entre indicateur de concession et rabais observé :",
      (periode.sConcession.notna() & periode.sConcession.ne(periode.avec_rabais.astype(int))).sum())

## 2. Fréquence et importance des concessions

Le rabais équivalent est calculé **pour chaque bail**, puis résumé par année. On distingue la fréquence des concessions de leur importance parmi les baux avec rabais. Les médianes annuelles des loyers décrivent le niveau; elles ne mesurent pas la hausse des mêmes appartements.

In [ ]:
resume = periode.groupby("annee").agg(
    nombre_baux=("hUnit", "size"),
    part_avec_rabais_pct=("avec_rabais", lambda x: 100 * x.mean()),
    loyer_contractuel_median=("sRent", "median"),
    loyer_effectif_median=("sRentEffective", "median"),
    rabais_moyen_tous_baux_pct=("rabais_pct", "mean"),
)
resume["rabais_moyen_baux_avec_rabais_pct"] = (
    periode.loc[periode.avec_rabais].groupby("annee").rabais_pct.mean()
)
display(resume.round(2))
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
resume.part_avec_rabais_pct.plot.bar(ax=axes[0], color="#4477AA")
axes[0].set(title="Fréquence des rabais", xlabel="Année de début du bail", ylabel="Baux avec rabais (%)")
resume[["rabais_moyen_tous_baux_pct", "rabais_moyen_baux_avec_rabais_pct"]].plot(ax=axes[1], marker="o")
axes[1].legend(["Tous les baux", "Baux avec rabais seulement"])
axes[1].set(title="Importance du rabais équivalent", xlabel="Année de début du bail", ylabel="Rabais moyen (%)")
plt.tight_layout()
plt.show()

## 3. Comparer la croissance sur les mêmes paires de baux

Nous associons chaque bail au bail immédiatement précédent du même appartement. Les deux croissances utilisent exactement les mêmes paires. Un bail invalide n'est pas supprimé avant cet appariement, afin de ne pas sauter artificiellement un bail intermédiaire.

Comme dans le notebook de départ, on retient les intervalles strictement compris entre 0,5 et 2,5 ans. La variation est annualisée : `(nouveau / précédent) ** (1 / durée_en_années) - 1`. Le groupe renouvellement/relocation est celui du nouveau bail.

In [ ]:
paires = base.sort_values(["hUnit", "sLeaseFrom"]).copy()
groupes = paires.groupby("hUnit")
for col in ["sRent", "sRentEffective", "sLeaseFrom", "valide"]:
    paires["precedent_" + col] = groupes[col].shift(1)
paires["duree_annees"] = (paires.sLeaseFrom - paires.precedent_sLeaseFrom).dt.days / 365.25
retenues = (paires.valide & paires.precedent_valide.eq(True)
            & paires.duree_annees.between(0.5, 2.5, inclusive="neither")
            & paires.annee.between(2021, 2025))
paires = paires.loc[retenues].copy()
paires["croissance_contractuelle_pct"] = 100 * ((paires.sRent / paires.precedent_sRent) ** (1 / paires.duree_annees) - 1)
paires["croissance_effective_pct"] = 100 * ((paires.sRentEffective / paires.precedent_sRentEffective) ** (1 / paires.duree_annees) - 1)
paires["rabais_actuel_pct"] = 100 * (1 - paires.sRentEffective / paires.sRent)
paires["rabais_precedent_pct"] = 100 * (1 - paires.precedent_sRentEffective / paires.precedent_sRent)
paires["variation_rabais_points"] = paires.rabais_actuel_pct - paires.rabais_precedent_pct
croissance = paires.groupby("annee").agg(
    nombre_paires=("hUnit", "size"),
    contractuelle_pct=("croissance_contractuelle_pct", "median"),
    effective_pct=("croissance_effective_pct", "median"),
    variation_mediane_rabais_points=("variation_rabais_points", "median"),
)
croissance["ecart_points"] = croissance.contractuelle_pct - croissance.effective_pct
display(croissance.round(2))
ax = croissance[["contractuelle_pct", "effective_pct"]].plot(marker="o")
ax.legend(["Loyer contractuel", "Loyer effectif"])
ax.axhline(0, color="gray", linewidth=1)
ax.axvline(2024, color="gray", linestyle="--", label="2024")
ax.set(title="Croissance annualisée : mêmes appartements, mêmes paires de baux", xlabel="Année du nouveau bail", ylabel="Croissance médiane (%)")
plt.tight_layout()
plt.show()

## 4. Distinguer renouvellements et relocations

La composition des baux peut aussi évoluer. On vérifie donc si la divergence apparaît dans chaque groupe. Les effectifs permettent de repérer les résultats reposant sur peu de comparaisons.

In [ ]:
par_type = paires.groupby(["annee", "type_bail"]).agg(
    nombre_paires=("hUnit", "size"),
    contractuelle_pct=("croissance_contractuelle_pct", "median"),
    effective_pct=("croissance_effective_pct", "median"),
    variation_mediane_rabais_points=("variation_rabais_points", "median"),
)
par_type["ecart_points"] = par_type.contractuelle_pct - par_type.effective_pct
display(par_type.round(2))
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, type_bail in zip(axes, ["Renouvellement", "Relocation"]):
    if type_bail in par_type.index.get_level_values("type_bail"):
        donnees = par_type.xs(type_bail, level="type_bail")
        donnees[["contractuelle_pct", "effective_pct"]].plot(ax=ax, marker="o")
        ax.legend(["Contractuel", "Effectif"])
    ax.axhline(0, color="gray", linewidth=1)
    ax.set(title=type_bail, xlabel="Année", ylabel="Croissance médiane annualisée (%)")
plt.tight_layout()
plt.show()

## 5. Interpréter les résultats

Pour chaque paire, le mécanisme est exact :

`nouveau effectif / ancien effectif = (nouveau contractuel / ancien contractuel) × (1 − nouveau rabais) / (1 − ancien rabais)`.

Si le rabais augmente, il réduit la croissance effective par rapport à la croissance contractuelle. Une concession déjà présente et de même proportion dans les deux baux ne crée pas à elle seule une divergence.

L'écart entre deux médianes n'est pas la médiane des écarts individuels. Les statistiques sur les rabais des mêmes paires complètent donc la comparaison des courbes.

In [ ]:
for annee in [2023, 2024, 2025]:
    if annee in croissance.index:
        ligne = croissance.loc[annee]
        display(Markdown(
            f"**{annee}** : {int(ligne.nombre_paires)} paires; croissance contractuelle médiane "
            f"**{ligne.contractuelle_pct:.2f} %**, effective **{ligne.effective_pct:.2f} %**; "
            f"écart **{ligne.ecart_points:.2f} %**."
        ))
comparaison_rabais = paires.assign(
    periode=np.where(paires.annee < 2024, "2021–2023", "2024–2025")
).groupby("periode").agg(
    nombre_paires=("hUnit", "size"),
    rabais_precedent_moyen_pct=("rabais_precedent_pct", "mean"),
    rabais_actuel_moyen_pct=("rabais_actuel_pct", "mean"),
    variation_moyenne_rabais_points=("variation_rabais_points", "mean"),
    part_rabais_en_hausse_pct=("variation_rabais_points", lambda x: 100 * x.gt(0.01).mean()),
)
display(comparaison_rabais.round(2))

### Conclusion : pourquoi la croissance contractuelle et la croissance effective divergent à partir de 2024

**1. Quand et de combien l'écart s'élargit-il?**
Jusqu'en 2023, les deux croissances évoluent ensemble : l'écart reste sous 1 point (0,91 en 2021, 0,81 en 2022, 0,27 en 2023).
Il passe à **1,62 point en 2024**, puis à **3,47 points en 2025** : le loyer inscrit au bail augmente de 7,04 %, le loyer
effectif de seulement 3,58 %. Environ la moitié de la hausse affichée en 2025 n'est pas perçue.

**2. Les concessions deviennent-elles plus fréquentes, plus généreuses, ou les deux?**
**Les deux.** La part des baux avec rabais passe de 54 % (2023) à 73 % (2024) puis **85 % (2025)**. Parmi les baux avec rabais,
le rabais moyen passe de 6,6 % à 7,8 % puis **10,7 %** du loyer contractuel. Sur l'ensemble des baux, le rabais moyen
passe de 3,6 % à **9,1 %** en deux ans.

**3. Sur les mêmes appartements, le rabais augmente-t-il par rapport au bail précédent?**
Oui, et c'est le mécanisme de la divergence. Pour chaque paire,
`croissance effective = croissance contractuelle × (1 − nouveau rabais) / (1 − ancien rabais)` : un rabais stable ne crée
aucun écart, seul un rabais **qui grossit** en crée un. Or :

- la hausse médiane du rabais d'un bail au suivant est nulle en 2023, puis de **1,16 point en 2024** et **2,97 points en 2025**,
  soit presque exactement l'écart observé entre les deux croissances;
- en 2024-2025, le rabais augmente pour **78 % des paires** (+2,67 points en moyenne), contre 53 % des paires (+0,41 point)
  en 2021-2023.

**4. Le phénomène concerne-t-il surtout les relocations ou aussi les renouvellements?**
**Les deux, presque à égalité.** En 2025, l'écart est de 3,24 points pour les relocations et de 3,50 points pour les
renouvellements, avec la même hausse médiane du rabais (environ 3 points). Les concessions ne servent donc pas seulement
à attirer de nouveaux locataires : elles servent aussi à **retenir** les locataires en place.

**En résumé :** à partir de 2024, Équinoxe continue d'augmenter les loyers inscrits aux baux, mais en redonne une part
croissante sous forme de concessions plus fréquentes et plus généreuses, sur les mêmes appartements, pour les nouveaux
comme pour les anciens locataires. La croissance contractuelle surestime donc la hausse réellement obtenue.

**Conséquence pour la prévision :** nous retenons la croissance du **loyer effectif**, qui tient compte des avantages
accordés. La question centrale pour 2026 devient l'évolution des concessions : si elles se stabilisent, la croissance
effective se rapprochera de la croissance contractuelle; si elles continuent d'augmenter, l'écart persistera.

**Limites :** les données internes montrent *comment* la divergence se produit, pas *pourquoi* le propriétaire accorde ces
rabais. Une explication par la concurrence ou par l'inoccupation du marché doit être appuyée par des données publiques
(SCHL). Ces résultats décrivent les baux de l'extrait et ne mesurent pas le revenu total du portefeuille.

## 6. Quels facteurs sont associés aux concessions ?

Nous cherchons des associations avec le **rabais équivalent en pourcentage**, plutôt qu'avec son montant en dollars (qui dépend directement du niveau du loyer).

### Peut-on utiliser le nombre d'unités vacantes ?

**Non, pas avec cet extrait.** Le notebook de départ précise que les volumes ne représentent pas le portefeuille réel et que l'occupation et l'inoccupation ne peuvent pas être calculées. `sAvailable` désigne une date de disponibilité : elle ne constitue pas un statut d'occupation observé. Un intervalle entre deux baux ne prouve pas non plus une période de vacance réelle.

Pour étudier l'inoccupation, il faudrait un historique fiable du nombre de logements vacants et du nombre total de logements, par immeuble et par mois. On pourrait alors rapprocher le taux d'inoccupation du mois précédent des concessions offertes aux nouveaux baux. Une série publique de la SCHL peut également servir de contexte de marché, mais ne mesure pas l'inoccupation d'Équinoxe.

Nous étudions ici les facteurs disponibles : superficie, nombre de chambres, étage, durée du bail, renouvellement/relocation, immeuble et année. Une corrélation décrit une association; elle ne démontre pas une cause.

In [ ]:
facteurs = {
    "sSqft": "Superficie (pi²)",
    "sBeds": "Nombre de chambres",
    "sFloor": "Étage",
    "sTermMonths": "Durée du bail (mois)",
    "sRenewal": "Renouvellement (1) / relocation (0)",
}
analyse = periode.copy()
for col in facteurs:
    analyse[col] = pd.to_numeric(analyse[col], errors="coerce")
colonnes = ["rabais_pct"] + list(facteurs)
analyse[colonnes] = analyse[colonnes].replace([np.inf, -np.inf], np.nan)

# Spearman : association entre les rangs, moins sensible aux valeurs extrêmes.
# Corrélation intra-groupe : Pearson après retrait des moyennes par immeuble/année.
# Les deux mesures répondent à des questions différentes : elles ne sont pas directement équivalentes.
resultats = []
for col, nom in facteurs.items():
    echantillon = analyse[["rabais_pct", col, "sBuilding", "annee"]].dropna().copy()
    rangs = echantillon[["rabais_pct", col]].rank()
    spearman = rangs.corr().loc["rabais_pct", col]
    g = echantillon.groupby(["sBuilding", "annee"])
    effectifs = g[col].transform("size")
    echantillon = echantillon.loc[effectifs >= 2].copy()
    centres = echantillon[["rabais_pct", col]] - echantillon.groupby(
        ["sBuilding", "annee"]
    )[["rabais_pct", col]].transform("mean")
    intra = centres.corr().loc["rabais_pct", col]
    resultats.append({
        "Facteur": nom,
        "Nombre de baux": len(rangs),
        "Spearman global": spearman,
        "Nombre de baux intra-groupe": len(centres),
        "Pearson au sein des immeubles/années": intra,
    })
correlations = pd.DataFrame(resultats).set_index("Facteur")
display(correlations.round(3))
ax = correlations[["Spearman global", "Pearson au sein des immeubles/années"]].plot.barh(figsize=(11, 5))
ax.axvline(0, color="black", linewidth=1)
ax.set(xlim=(-1, 1), xlabel="Coefficient de corrélation", ylabel="", title="Facteurs associés au rabais équivalent")
ax.legend(loc="best")
plt.tight_layout()
plt.show()

**Comment lire le tableau :** un coefficient positif indique que le rabais tend à augmenter avec le facteur; un coefficient négatif indique qu'il tend à diminuer. Un coefficient proche de zéro signifie une faible association selon cette mesure, mais n'exclut pas une relation plus complexe. Pour `sRenewal`, une association négative signifie que les renouvellements tendent à recevoir moins de rabais.

Les corrélations globales peuvent refléter la composition des immeubles et des années. La seconde colonne mesure l'association linéaire entre baux d'un même immeuble et d'une même année, en retirant les moyennes de ces groupes. Elle ne neutralise pas tous les facteurs et ne constitue pas une preuve causale. Les baux répétés d'un même appartement ne sont pas indépendants; nous ne présentons donc pas de tests de significativité naïfs.

In [ ]:
def resume_concessions(df, cles):
    tableau = df.groupby(cles, observed=True).agg(
        nombre_baux=("hUnit", "size"),
        nombre_appartements=("hUnit", "nunique"),
        part_avec_rabais_pct=("avec_rabais", lambda x: 100 * x.mean()),
        rabais_moyen_pct=("rabais_pct", "mean"),
        rabais_median_pct=("rabais_pct", "median"),
    )
    tableau["rabais_moyen_si_rabais_pct"] = (
        df.loc[df.avec_rabais].groupby(cles, observed=True).rabais_pct.mean()
    )
    return tableau

par_chambres = resume_concessions(analyse, ["sBeds"])
par_bail = resume_concessions(analyse, ["annee", "type_bail"])
par_immeuble = resume_concessions(analyse, ["sBuilding", "annee"])
print("Concessions par nombre de chambres")
display(par_chambres.round(2))
print("Concessions par année et type de bail")
display(par_bail.round(2))
print("Concessions par immeuble et année")
display(par_immeuble.round(2))

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
par_chambres.rabais_moyen_pct.plot.bar(ax=axes[0], color="#4477AA")
axes[0].set(title="Selon le nombre de chambres", xlabel="Nombre de chambres", ylabel="Rabais moyen (%)")
par_bail.rabais_moyen_pct.unstack("type_bail").plot(ax=axes[1], marker="o")
axes[1].set(title="Selon le type de bail", xlabel="Année", ylabel="Rabais moyen (%)")
par_immeuble.rabais_moyen_pct.unstack("sBuilding").plot(ax=axes[2], marker="o")
axes[2].set(title="Selon l'immeuble", xlabel="Année", ylabel="Rabais moyen (%)")
axes[2].legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Nuage de points : conserve tous les baux valides avec une superficie renseignée.
fig, ax = plt.subplots(figsize=(10, 5))
for type_bail, groupe in analyse.groupby("type_bail"):
    ax.scatter(groupe.sSqft, groupe.rabais_pct, alpha=0.15, s=15, label=type_bail)
ax.set(title="Superficie et rabais équivalent", xlabel="Superficie (pi²)", ylabel="Rabais équivalent (%)")
ax.legend()
plt.tight_layout()
plt.show()

# Résumé automatique descriptif, sans conclure à un effet causal.
classement = correlations["Spearman global"].dropna().abs().sort_values(ascending=False)
if not classement.empty:
    facteur = classement.index[0]
    coefficient = correlations.loc[facteur, "Spearman global"]
    display(Markdown(
        f"Parmi les facteurs numériques examinés, **{facteur}** présente la corrélation "
        f"globale de Spearman la plus élevée en valeur absolue avec le rabais "
        f"(**{coefficient:.3f}**). Comparer ce résultat à l'association au sein des "
        "immeubles/années et aux effectifs avant de l'interpréter."
    ))

### Conclusion : quels facteurs sont associés aux concessions?

**Les caractéristiques des appartements n'expliquent presque rien.**

- Toutes les corrélations globales sont faibles (|coefficient| ≤ 0,11). Les plus fortes sont négatives : étage (−0,114) et
  superficie (−0,109).
- **Au sein d'un même immeuble et d'une même année, elles disparaissent** (toutes entre −0,02 et +0,04). Les faibles
  corrélations globales reflètent donc la **composition** : les nouveaux immeubles, aux unités plus petites, sont arrivés
  au moment où les rabais augmentaient. Une grande ou une petite unité, à un étage haut ou bas, reçoit à peu près
  le même rabais que ses voisines la même année.
- Par nombre de chambres, les écarts sont modestes (rabais moyen de 5,1 % à 6,4 %), et nuls parmi les baux qui ont un rabais
  (8,1 % à 8,4 %).
- **Renouvellement ou relocation : aucune différence.** Parmi les baux avec rabais, le rabais moyen est identique pour les deux
  types chaque année (par exemple 10,73 % contre 10,67 % en 2025).
- La durée du bail n'est pas associée au rabais (coefficient d'environ 0).

**Ce qui compte, c'est l'année, et un peu l'immeuble.**

- Le rabais suit d'abord le **temps** : dans chaque immeuble, il reste stable jusqu'en 2023 puis monte fortement en 2024
  et 2025. Le phénomène touche **tous les immeubles**, anciens immeubles de Laval compris.
- Les nouveaux immeubles haut de gamme sont légèrement au-dessus : en 2025, la part des baux avec rabais atteint
  97,5 % à The Met, 94 % à Westpark et 93 % au Carlyle, contre 75 à 83 % dans les immeubles de Laval.

**Interprétation :** les concessions se comportent comme une **politique commerciale globale**, qui dépend de l'année (donc
probablement du contexte de marché), et non comme une réponse aux caractéristiques d'un appartement particulier.
Les corrélations décrivent des associations, pas des causes, et les baux successifs d'un même appartement ne sont pas
indépendants.

**Limite concernant l'inoccupation :** aucune corrélation avec le nombre d'unités vacantes n'est calculée, faute de mesure
fiable dans les fichiers. L'hypothèse d'un lien avec le marché reste à tester avec une série publique (taux d'inoccupation
de la SCHL pour les RMR de Montréal et d'Ottawa), à la bonne période et à la bonne échelle géographique.